# colab_clue_rl

> Phase 12, Track B on a GPU (`specs/multiagent/phase12/`), Phase 13's Track B runs (`specs/multiagent/phase13/`), Phase 16's extra seeds (`specs/multiagent/phase16/`) Phase 17's team-size runs (`specs/multiagent/phase17/`) and Phase 18's clone runs on the colony board (`specs/multiagent/phase18/`): the warm start and the RL runs of a 3B open-weight model on the Phase 11 clue board, as a standalone Google Colab notebook (an A100 runtime; any Linux GPU machine works the same way).

[Open in Colab](https://colab.research.google.com/github/Positron00/explRL/blob/main/nbs/colab_clue_rl.ipynb). Outputs go to Google Drive (`MyDrive/explRL_rl/`); nothing here needs an API key or the repo's `results/`. Run the sections in order, one session each where noted; every section resumes after a disconnection.

1. **Set-up**: GPU check, install, Drive.
2. **Pilot** (about 1 hour): a few warm-start and RL steps; step time and memory, so that each main run fits a few hours.
3. **Warm start** (about 1.6 hours): supervised training on the rule agent's moves; the adapter is checked on the Mac (`demo_rl`).
4. **RL speed check** (session 2, about 15 minutes): three steps with the final settings, before `RL_PLAN` is frozen.
5. **Learning-rate pilot** (session 3, about 1.5 hours): three fixed learning rates for ten steps, before `RL_PLAN` v2 is frozen.
6. **RL run** (1.5-2 hours, one session per reward scheme): set `SCHEME`, run; needs `RL_PLAN` frozen.
7. **Evaluation** (after the three runs, about 1.5-2 hours): the five models on the evaluation boards (cross-play with Haiku runs on the Mac).
8. **Phase 13 RL run** (about 3 hours, one session per run): set `ARM` and `SEED` in the configuration cell, run; needs `RL13_PLAN` frozen.
9. **Phase 13 evaluation** (after the five Phase 13 runs, about 2 hours): the five new adapters on the evaluation boards, without cross-play (on the Mac).
10. **Phase 16: runs, evaluation and pack in one session** (`specs/multiagent/phase16/`; one cell, about 20 hours, no manual steps): six more training seeds (`individual` 2-4, `indpen` 3-5), their evaluation on the main boards and `explRL_rl16.zip`; needs `RL16_PLAN` frozen; resumes after a disconnect (restart, set-up, configuration, the same cell).
11. **Phase 17 step 0** (`specs/multiagent/phase17/`; one cell, about 8 hours, after Phase 16): every trained adapter on the team-size grid (4-32 agents, clean and with liars) -> `explRL_rl17s0.zip`.
12. **Phase 17 pilot** (one cell, about 50 hours): four 30-step recipes and the selection printout -> `explRL_rl17pilot3.zip` (plan v3: loss and KL on value tokens, a per-token clip, a 3,072-token training window; v1 and v2 in `runs_p17pilot` and `runs_p17pilot2`); needs `PILOT17_PLAN` frozen.
13. **Phase 17 main** (one cell per account; 8 runs of 60 steps, about 86 A100 hours plus evaluation): claims runs (set `HOST` per account), trains them, then one account evaluates -> `explRL_rl17.zip`; needs `RL17_PLAN` frozen with the recipe and the confirmed hours.
14. **Phase 18 speed check** (`specs/multiagent/phase18/`, Part B; one account, about 1 hour): seconds per step, the training window and projected hours for `RL18_PLAN`.
15. **Phase 18 main** (one cell per account): claims the 5 runs (Part B lite: team reward on the colony board, 30 steps), trains them, then both accounts evaluate (models claimed one at a time) -> `explRL_rl18.zip`; needs `RL18_PLAN` frozen and confirmed.
16. **Pack**: a zip of the adapters, metrics and configurations to download to `results/multiagent/rl/colab/` on the Mac.

**After any error, use Runtime -> Restart session before running anything else:** a failed cell keeps its models in GPU memory (and vLLM's engine runs in a separate process that keeps its share), so the next cell would run out of memory. Then rerun set-up and configuration.

The sections are behind the `gpu` test flag, so normal tests only run the configuration cell.

In [ ]:
#| gpu
import subprocess, sys
if 'google.colab' in sys.modules:
    # Colab preinstalls an older torchao; the PEFT version of the [train] extra needs torchao >= 0.16
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchao>=0.16.0'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'explRL[train] @ git+https://github.com/Positron00/explRL'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/explRL_rl'
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU: use a GPU runtime')

In [ ]:
import json
import time
from pathlib import Path

from explRL.coop_grid import MODELS, RL13_PLAN, RL_PLAN, coopHash
from explRL.rl16_grid import RL16_PLAN
from explRL.rl17_grid import PILOT17_PLAN, RL17_PLAN
from explRL.rl18_grid import RL18_PLAN
from explRL.train import demo_artifact_dir

OUT = Path(globals().get('OUT') or demo_artifact_dir('colab', 'multiagent/rl'))
BASE = MODELS['base']
SCHEME = 'credit'                      # set before each RL session: 'individual', 'team' or 'credit'
ARM, SEED = 'credit', 2                # Phase 13: set before each session, one of RL13_PLAN's runs (arm, seed)
RUNS = OUT / ('runs' if RL_PLAN['version'] == 1 else f"runs_v{RL_PLAN['version']}")   # v1 runs stay in runs/
SFT_SEEDS = range(RL_PLAN['sftData']['seeds'][0], RL_PLAN['sftData']['seeds'][1] + 1)
RUNS13 = OUT / 'runs_p13'              # Phase 13 runs: runs_p13/<arm>_s<seed>/
RUNS16 = OUT / 'runs_p16'              # Phase 16 runs: runs_p16/<arm>_s<seed>/
RUNS17 = OUT / 'runs_p17'              # Phase 17 main runs: runs_p17/<arm>_s<seed>/
RUNS18 = OUT / 'runs_p18'              # Phase 18 Part B runs: runs_p18/<arm>_s<seed>/
HOST = 'account-a'                     # Phase 17: a different name on each Colab account sharing the Drive folder
TAKEOVER = []                          # Phase 17: runs this account may take over from another (stop that runtime first)
print(f"RL_PLAN v{RL_PLAN['version']} {coopHash(RL_PLAN)} (frozen: {RL_PLAN['frozen']}); base model {BASE}; RL runs in {RUNS}")
print(f"RL13_PLAN v{RL13_PLAN['version']} {coopHash(RL13_PLAN)} (frozen: {RL13_PLAN['frozen']}); Phase 13 run {ARM}_s{SEED} in {RUNS13}")
print(f"RL16_PLAN v{RL16_PLAN['version']} {coopHash(RL16_PLAN)} (frozen: {RL16_PLAN['frozen']}); Phase 16 runs in {RUNS16}")
print(f"PILOT17_PLAN {coopHash(PILOT17_PLAN)} (frozen: {PILOT17_PLAN['frozen']}); RL17_PLAN {coopHash(RL17_PLAN)} (frozen: {RL17_PLAN['frozen']}); host {HOST}")
print(f"RL18_PLAN {coopHash(RL18_PLAN)} (frozen: {RL18_PLAN['frozen']}); Phase 18 runs in {RUNS18}")


def stream(code):
    "Run `code` in a fresh Python process, echoing its output here (GPU memory is freed when it ends); -> its exit code."
    import os, subprocess, sys
    p = subprocess.Popen([sys.executable, '-u', '-c', code], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    while chunk := os.read(p.stdout.fileno(), 4096):
        sys.stdout.write(chunk.decode(errors='replace'))
        sys.stdout.flush()
    return p.wait()


def packZip(name, dirs):
    "OUT / name: the files under `dirs` (no ckpt-* checkpoints)."
    import zipfile
    with zipfile.ZipFile(OUT / name, 'w') as z:
        for p in sorted(OUT.rglob('*')):
            rel = p.relative_to(OUT)
            if p.is_file() and rel.parts[0] in dirs and 'ckpt-' not in str(rel):
                z.write(p, rel)
    print(OUT / name, round((OUT / name).stat().st_size / 1e6, 1), 'MB')

## Pilot (`gpu`; session 1 only, already done): step time and memory

In [ ]:
#| gpu
from explRL.clue_rl import RLConfig, trainRL
from explRL.clue_sft import SFTConfig, mergeAdapter, sftExamples, trainSFT
P = OUT / 'pilot'
t0 = time.time()
adapter = trainSFT(SFTConfig(BASE, str(P / 'sft'), maxSteps=20), sftExamples(range(20000, 20010), turns=20))
t1 = time.time()
mergeAdapter(BASE, adapter, P / 'warm')
trainRL(RLConfig(str(P / 'warm'), str(P / 'rl'), scheme=SCHEME, **{**RL_PLAN['rl'], 'steps': 3, 'ckptEvery': 1}))
steps = [json.loads(x) for x in open(P / 'rl' / 'metrics.jsonl')]
for x in steps:
    print(f"RL step {x['step']}: rollout {x['rolloutSeconds']:.0f} s, update {x['updateSeconds']:.0f} s, trained turns {x['samples']}")
per = sum(x['rolloutSeconds'] + x['updateSeconds'] for x in steps) / len(steps)
print(f"warm start: {(t1 - t0) / 20:.1f} s per step; RL: {per:.0f} s per step -> {per * RL_PLAN['rl']['steps'] / 3600:.1f} h "
      f"per run; peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB; session {(time.time() - t0) / 3600:.2f} h")

## Warm start (`gpu`)

In [ ]:
#| gpu
from explRL.clue_sft import SFTConfig, sftExamples, trainSFT
t0 = time.time()
ex = sftExamples(SFT_SEEDS, sizes=tuple(RL_PLAN['sftData']['sizes']), turns=RL_PLAN['sftData']['turns'])
adapter = trainSFT(SFTConfig(BASE, str(OUT / 'warm'), **RL_PLAN['sft']), ex)
print(f'{len(ex)} examples; adapter {adapter}; {(time.time() - t0) / 3600:.2f} GPU hours. Next: download it and run the '
      'warm-start check on the Mac (demo_rl).')

## RL speed check (`gpu`; start of session 2, before `RL_PLAN` is frozen)

Three RL steps from the warm-start model with `RL_PLAN`'s settings (a quarter of the turns trained on, gradient checkpointing off): the time per step without vLLM's start-up, the projected hours per run, and the peak memory. Send the printout to Claude; the step count is then fixed and `RL_PLAN` frozen.

In [ ]:
#| gpu
import shutil
import numpy as np
from explRL.clue_rl import RLConfig, trainRL
from explRL.clue_sft import mergeAdapter
WARM = OUT / 'warm_model'
if not (WARM / 'config.json').exists():
    mergeAdapter(BASE, OUT / 'warm' / 'adapter', WARM)
SC = OUT / 'speedcheck'
shutil.rmtree(SC, ignore_errors=True)
trainRL(RLConfig(str(WARM), str(SC), scheme=SCHEME, **{**RL_PLAN['rl'], 'steps': 3, 'ckptEvery': 3}))
steps = [json.loads(x) for x in open(SC / 'metrics.jsonl')]
for x in steps:
    print(f"step {x['step']}: rollout {x['rolloutSeconds']:.0f} s, update {x['updateSeconds']:.0f} s, trained turns "
          f"{x['samples']} of {x['signalTurns']}, invalid {x['invalid']:.3f}, KL {x['kl']:.4f}, "
          f"gradient checkpointing {'on (out-of-memory fallback)' if x['gradCkpt'] else 'off'}")
per = np.mean([x['rolloutSeconds'] + x['updateSeconds'] for x in steps[1:]])     # step 1 includes vLLM's start-up
print(f"steady state {per:.0f} s per step -> {per * 40 / 3600:.1f} h for 40 steps, {per * 60 / 3600:.1f} h for 60; "
      f"peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## Learning-rate pilot (`gpu`; session 3, before `RL_PLAN` v2 is frozen; about 1.5 hours)

The v1 runs (learning rate 1e-5) barely moved the model. Ten steps under `credit` at each of three fixed learning rates (the controller off), from the warm-start model: the per-step policy change (`stepKL`), the KL to the warm-start model, invalid replies, welfare and time per step. Send the printout to Claude; the starting and maximum learning rates are then set and `RL_PLAN` v2 frozen. A run already done is not redone.

In [ ]:
#| gpu
import numpy as np
from explRL.clue_rl import RLConfig, trainRL
from explRL.clue_sft import mergeAdapter
WARM = OUT / 'warm_model'
if not (WARM / 'config.json').exists():
    mergeAdapter(BASE, OUT / 'warm' / 'adapter', WARM)
lp = RL_PLAN['lrPilot']
for lr in lp['lrs']:
    d = OUT / 'lrpilot' / f'lr{lr:g}'
    t0 = time.time()
    trainRL(RLConfig(str(WARM), str(d), scheme=lp['scheme'], **{**RL_PLAN['rl'], 'lr': lr, 'klTarget': 0.0, 'warmup': 0,
                                                                 'steps': lp['steps'], 'ckptEvery': lp['steps']}))
    m = [json.loads(x) for x in open(d / 'metrics.jsonl')]
    print(f"lr {lr:g}: per-step KL mean {np.mean([x['stepKL'] for x in m]):.4f} max {max(x['stepKL'] for x in m):.4f}; "
          f"KL to warm start, last step {m[-1]['kl']:.4f}; invalid {np.mean([x['invalid'] for x in m]):.3f}; welfare first 3 "
          f"{np.mean([x['welfare'] for x in m[:3]]):.3f}, last 3 {np.mean([x['welfare'] for x in m[-3:]]):.3f}; "
          f"{np.mean([x['rolloutSeconds'] + x['updateSeconds'] for x in m[1:]]):.0f} s per step; this run {(time.time() - t0) / 3600:.2f} h")

## RL run (`gpu`; one session per reward scheme)

In [ ]:
#| gpu
from explRL.clue_rl import RLConfig, trainRL
from explRL.clue_sft import mergeAdapter
assert RL_PLAN['frozen'], 'freeze and commit RL_PLAN (after the pilot and the warm-start check) before the main runs'
WARM = OUT / 'warm_model'
if not (WARM / 'config.json').exists():
    mergeAdapter(BASE, OUT / 'warm' / 'adapter', WARM)
t0 = time.time()
final = trainRL(RLConfig(str(WARM), str(RUNS / SCHEME), scheme=SCHEME, **RL_PLAN['rl']))
print(f'{SCHEME}: {final}; this session {(time.time() - t0) / 3600:.2f} GPU hours')

## Evaluation (`gpu`; after the three RL runs; about 1.5-2 hours)

`RL_PLAN`'s evaluation boards (40 turns, held-out seeds) for the five models: the untrained base model, the warm-start model and the three RL adapters on it. Cross-play with Claude Haiku needs the API key, which stays off Colab: it runs on the Mac afterwards (`demo_rl`), which merges it with these rows. Each model's rows are saved to Drive when it finishes, so after a disconnect a rerun continues with the next model. Then run Pack.

In [ ]:
#| gpu
import numpy as np
from explRL.bench_grid import rowKey, writeCSV
from explRL.clue_roll import evalModels
from explRL.clue_sft import mergeAdapter
from explRL.coop_grid import evalSuite
assert RL_PLAN['frozen'], 'the evaluation uses the frozen RL_PLAN'
WARM = OUT / 'warm_model'
if not (WARM / 'config.json').exists():
    mergeAdapter(BASE, OUT / 'warm' / 'adapter', WARM)
E = OUT / 'eval'
cells = [c for c in evalSuite() if not c['level'].endswith(('cross1', 'cross3'))]      # cross-play (Haiku): on the Mac
models = dict(untrained=(BASE, None), warm=(str(WARM), None),
              **{sc: (str(WARM), str(RUNS / sc / 'final')) for sc in RL_PLAN['schemes']})
t0 = time.time()
rows = evalModels(cells, models, E, cache='/content/evalcache', logDir=E / 'turns')
writeCSV(rows, E / 'runs.csv')
for m in models:
    w = [float(r['welfare']) for r in rows if r['level'] == f'{m}|main']
    print(f"{m:10s} main boards: welfare per agent-turn {np.mean(w):.3f} ({len(w)} boards)")
print(f'{len(rows)} boards in all; this session {(time.time() - t0) / 3600:.2f} GPU hours. Next: Pack.')

## Phase 13 RL run (`gpu`; one session per run, about 3 hours)

One of `RL13_PLAN`'s five runs per session (`credit` seed 2, `indpen` seeds 1-2, `credturn` seeds 1-2; seed 1 = `trainSeed` 0, seed 2 = `trainSeed` 1): `RL_PLAN` v2's settings with the arm's changes (`indpen`: individual reward, a training-only cost of 2.5 per wrong answer, roll-back on a jump above 15% invalid replies; `credturn`: credit reward, per-turn advantages, the same cost and roll-back). Set `ARM` and `SEED` in the configuration cell first (restart the session between runs). A run resumes from its last checkpoint after a disconnection; a halted run (stopping rule or two roll-backs in a row) is not resumed. Note the GPU hours printed at the end.

In [ ]:
#| gpu
from explRL.clue_rl import trainRL
from explRL.clue_sft import mergeAdapter
from explRL.coop_grid import RL13_RUNS, rl13Config, rl13Name
assert RL13_PLAN['frozen'], 'freeze and commit RL13_PLAN before the Phase 13 runs'
assert (ARM, SEED) in RL13_RUNS, f'{ARM}_s{SEED} is not one of the Phase 13 runs {RL13_RUNS}'
WARM = OUT / 'warm_model'
if not (WARM / 'config.json').exists():
    mergeAdapter(BASE, OUT / 'warm' / 'adapter', WARM)
t0 = time.time()
final = trainRL(rl13Config(ARM, SEED, str(WARM), str(RUNS13 / rl13Name(ARM, SEED))))
print(f'{rl13Name(ARM, SEED)}: {final}; this session {(time.time() - t0) / 3600:.2f} GPU hours')

## Phase 13: all five runs in one session (`gpu`; about 15 hours; instead of the per-run section above)

Trains `RL13_PLAN`'s five runs one after another, each in its own Python process, so each run's GPU memory (vLLM's engine included) is freed when it ends and no restart is needed between runs. Runs already finished (a `final` adapter or `halted.json`) are skipped, and an unfinished one resumes from its last checkpoint, so after a disconnect: restart, set-up, configuration, then this cell again. `ARM` and `SEED` are not used here. Each run's output streams below (progress bar and one line per step); a run that fails stops the batch. To have no further manual steps, queue the Phase 13 evaluation and Pack cells after this one (Runtime -> Run after).

In [ ]:
#| gpu
import os
import sys
import subprocess
from explRL.coop_grid import RL13_RUNS, rl13Name
assert RL13_PLAN['frozen'], 'freeze and commit RL13_PLAN before the Phase 13 runs'
ORDER = [('credturn', 1), ('indpen', 1), ('credturn', 2), ('indpen', 2), ('credit', 2)]   # the main test first
assert sorted(ORDER) == sorted(RL13_RUNS), (ORDER, RL13_RUNS)
WARM = OUT / 'warm_model'


def stream(code):
    "Run `code` in a fresh Python process, echoing its output (progress bar included) here; -> its exit code."
    p = subprocess.Popen([sys.executable, '-u', '-c', code], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    while chunk := os.read(p.stdout.fileno(), 4096):
        sys.stdout.write(chunk.decode(errors='replace'))
        sys.stdout.flush()
    return p.wait()


if not (WARM / 'config.json').exists():
    assert stream(f"from explRL.clue_sft import mergeAdapter; mergeAdapter({BASE!r}, {str(OUT / 'warm' / 'adapter')!r}, "
                  f"{str(WARM)!r})") == 0, 'merging the warm-start model failed'
for arm, seed in ORDER:
    name, d = rl13Name(arm, seed), RUNS13 / rl13Name(arm, seed)
    if (d / 'final').exists() or (d / 'halted.json').exists():
        print(f'{name}: already done, skipped')
        continue
    print(f'=== {name} ===', flush=True)
    t0 = time.time()
    rc = stream(f"from explRL.clue_rl import trainRL; from explRL.coop_grid import rl13Config; "
                f"print(trainRL(rl13Config({arm!r}, {seed}, {str(WARM)!r}, {str(d)!r})))")
    print(f'{name}: exit {rc}, {(time.time() - t0) / 3600:.2f} GPU hours', flush=True)
    if rc:
        raise RuntimeError(f'{name} failed (exit {rc}): restart the session, run set-up and configuration, then this cell')
print('all five Phase 13 runs done. Next: Phase 13 evaluation, then Pack.')

## Phase 13 evaluation (`gpu`; after the five Phase 13 runs; about 2 hours)

`RL13_PLAN`'s evaluation on Colab: the five new adapters on `RL_PLAN`'s main, generalisation and liar boards (no cross-play; the Phase 12 warm-start and v2 rows are reused, not rerun). Cross-play for both seeds of the best arm runs on the Mac (`demo_rl`), which merges it with these rows. Each adapter's rows are saved to Drive when it finishes, so after a disconnect a rerun continues with the next one. Then run Pack.

In [ ]:
#| gpu
import numpy as np
from explRL.bench_grid import writeCSV
from explRL.clue_roll import evalModels
from explRL.clue_sft import mergeAdapter
from explRL.coop_grid import RL13_RUNS, rl13Name, rl13Suite
assert RL13_PLAN['frozen'], 'the evaluation uses the frozen RL13_PLAN'
WARM = OUT / 'warm_model'
if not (WARM / 'config.json').exists():
    mergeAdapter(BASE, OUT / 'warm' / 'adapter', WARM)
E13 = OUT / 'eval_p13'
names = [rl13Name(a, s) for a, s in RL13_RUNS]
models = {m: (str(WARM), str(RUNS13 / m / 'final')) for m in names}
t0 = time.time()
rows = evalModels(rl13Suite(names), models, E13, cache='/content/evalcache', logDir=E13 / 'turns')
writeCSV(rows, E13 / 'runs.csv')
for m in names:
    w = [float(r['welfare']) for r in rows if r['level'] == f'{m}|main']
    print(f"{m:12s} main boards: welfare per agent-turn {np.mean(w):.3f} ({len(w)} boards)")
print(f'{len(rows)} boards in all; this session {(time.time() - t0) / 3600:.2f} GPU hours. Next: Pack.')

## Phase 16: runs, evaluation and pack in one session (`gpu`; about 20 hours; no manual steps)

After set-up and configuration, this one cell does all of Phase 16 (`RL16_PLAN`): (1) trains the six new runs in an interleaved order (`individual_s2`, `indpen_s3`, `individual_s3`, `indpen_s4`, `individual_s4`, `indpen_s5`), each in its own Python process so its GPU memory (vLLM's engine included) is freed when it ends; (2) evaluates the six adapters on the 15 main boards in its own process; (3) writes `explRL_rl16.zip` (`runs_p16`, `eval_p16`; Phase 13's `explRL_rl.zip` is left as it is) to Google Drive; (4) prints each adapter's welfare, clues shared per clue held and solo share. Finished or halted runs and evaluated adapters are skipped, and an unfinished run resumes from its last checkpoint, so after a disconnect: restart, set-up, configuration, then this cell again. A 20-hour batch needs background execution (Colab Pro+) or a tab that stays connected.

In [ ]:
#| gpu
import os
import subprocess
import sys
import zipfile
import numpy as np
from explRL.bench_grid import readCSV
from explRL.rl16_grid import RL16_PLAN, RL16_RUNS, rl16Name
assert RL16_PLAN['frozen'], 'freeze, commit and push RL16_PLAN before the Phase 16 runs'
ORDER = list(RL16_RUNS)                                     # interleaved: individual and indpen alternate
WARM, E16 = OUT / 'warm_model', OUT / 'eval_p16'
RESUME = 'restart the session, run set-up and configuration, then this cell again (finished steps are skipped)'


def stream(code):
    "Run `code` in a fresh Python process, echoing its output (progress bar included) here; -> its exit code."
    p = subprocess.Popen([sys.executable, '-u', '-c', code], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    while chunk := os.read(p.stdout.fileno(), 4096):
        sys.stdout.write(chunk.decode(errors='replace'))
        sys.stdout.flush()
    return p.wait()


t00 = time.time()
if not (WARM / 'config.json').exists():
    assert stream(f"from explRL.clue_sft import mergeAdapter; mergeAdapter({BASE!r}, {str(OUT / 'warm' / 'adapter')!r}, "
                  f"{str(WARM)!r})") == 0, f'merging the warm-start model failed: {RESUME}'
# 1. training, one process per run
for arm, seed in ORDER:
    name, d = rl16Name(arm, seed), RUNS16 / rl16Name(arm, seed)
    if (d / 'final').exists() or (d / 'halted.json').exists():
        print(f'{name}: already done, skipped')
        continue
    print(f'=== {name} ===', flush=True)
    t0 = time.time()
    rc = stream(f"from explRL.clue_rl import trainRL; from explRL.rl16_grid import rl16Config; "
                f"print(trainRL(rl16Config({arm!r}, {seed}, {str(WARM)!r}, {str(d)!r})))")
    print(f'{name}: exit {rc}, {(time.time() - t0) / 3600:.2f} GPU hours', flush=True)
    if rc:
        raise RuntimeError(f'{name} failed (exit {rc}): {RESUME}')
names = [rl16Name(a, s) for a, s in ORDER]
missing = [m for m in names if not (RUNS16 / m / 'final').exists()]
if missing:
    raise RuntimeError(f'no final adapter for {missing}: {RESUME}')
for m in names:
    if (RUNS16 / m / 'halted.json').exists():
        print(f"{m}: halted ({json.loads((RUNS16 / m / 'halted.json').read_text())['reason']}); evaluated from its final adapter")
# 2. evaluation, in its own process (adapters already evaluated are read back)
models = {m: (str(WARM), str(RUNS16 / m / 'final')) for m in names}
rc = stream("from explRL.bench_grid import writeCSV; from explRL.clue_roll import evalModels; from explRL.rl16_grid import rl16Suite; "
            f"rows = evalModels(rl16Suite({names!r}), {models!r}, {str(E16)!r}, cache='/content/evalcache', "
            f"logDir={str(E16 / 'turns')!r}); writeCSV(rows, {str(E16 / 'runs.csv')!r}); print(len(rows), 'boards')")
if rc:
    raise RuntimeError(f'evaluation failed (exit {rc}): {RESUME}')
# 3. pack to Google Drive (its own zip: Phase 13's explRL_rl.zip is left as it is)
with zipfile.ZipFile(OUT / 'explRL_rl16.zip', 'w') as z:
    for p in sorted(OUT.rglob('*')):
        rel = p.relative_to(OUT)
        if p.is_file() and rel.parts[0] in ('runs_p16', 'eval_p16') and 'ckpt-' not in str(rel):
            z.write(p, rel)
# 4. summary
rows = readCSV(E16 / 'runs.csv')
for m in names:
    rs = [r for r in rows if r['level'] == f'{m}|main']
    v = lambda k: np.nanmean([float(r[k]) for r in rs])
    print(f"{m:14s} welfare {v('welfare'):.3f}  shared per held {v('sharePerHeld'):.2f}  solo {v('soloShare'):.2f}  ({len(rs)} boards)")
print(f"{OUT / 'explRL_rl16.zip'} ({(OUT / 'explRL_rl16.zip').stat().st_size / 1e6:.1f} MB); this session {(time.time() - t00) / 3600:.2f} "
      'hours. Next, on the Mac: download the zip, unzip into results/multiagent/rl/colab/, run demo_rl16 (flag train).')

## Phase 17 step 0: every trained adapter on the team-size grid (`gpu`; one cell; about 8 hours)

The warm start and the 14 Phase 12-16 adapters on `teamSizeSuite` (clean boards at 4, 8, 16 and 32 agents, and boards with a quarter of the agents liars), 40 turns, in one process; models already evaluated are read back after a disconnect. Writes `eval_p17s0/` and `explRL_rl17s0.zip`. An L4 runtime also works.

Models are claimed one at a time, so two accounts can share step 0 (run the cell on both); models not yet on Drive are skipped; the account that finds all 15 evaluated writes `runs.csv` and the zip.

In [ ]:
#| gpu
from explRL.rl17_grid import STEP0_MODELS, claim
WARM, E0 = OUT / 'warm_model', OUT / 'eval_p17s0'
if not (WARM / 'config.json').exists():
    assert stream(f"from explRL.clue_sft import mergeAdapter; mergeAdapter({BASE!r}, {str(OUT / 'warm' / 'adapter')!r}, {str(WARM)!r})") == 0
models = {m: (str(WARM), None if d is None else str(OUT / d / 'final')) for m, d in STEP0_MODELS.items()}
missing = [m for m, (_, a) in models.items() if a and not Path(a).exists()]
if missing:
    print(f'not on Drive yet (Phase 16 still running?): {missing}; rerun this cell later for these')
t0 = time.time()
for m, v in models.items():                                    # one model at a time, claimed, so two accounts can share step 0
    if m in missing or (E0 / f'{m}.csv').exists() or not claim(E0 / 'claims' / m, HOST, TAKEOVER):
        continue
    print(f'=== {m} on {HOST} ===', flush=True)
    rc = stream("from explRL.clue_roll import evalModels; from explRL.rl17_grid import teamSizeSuite; "
                f"evalModels(teamSizeSuite([{m!r}]), {{{m!r}: {v!r}}}, {str(E0)!r}, cache='/content/evalcache', logDir={str(E0 / 'turns')!r})")
    if rc:
        raise RuntimeError(f'step 0 failed on {m} (exit {rc}): ' + 'restart the session, run set-up and configuration, then this cell again (finished steps are skipped)')
left = [m for m in models if not (E0 / f'{m}.csv').exists()]
if left:
    print(f'step 0 not complete yet: {left} (other account, or not on Drive yet): rerun this cell later')
elif claim(E0 / 'claims' / '_pack', HOST, TAKEOVER):
    assert stream("from explRL.bench_grid import writeCSV; from explRL.clue_roll import evalModels; from explRL.rl17_grid import teamSizeSuite; "
                  f"rows = evalModels(teamSizeSuite({list(models)!r}), {models!r}, {str(E0)!r}); "
                  f"writeCSV(rows, {str(E0 / 'runs.csv')!r}); print(len(rows), 'boards')") == 0       # all read back
    packZip('explRL_rl17s0.zip', ('eval_p17s0',))
print(f'step 0 on {HOST}: {(time.time() - t0) / 3600:.2f} hours.')

## Phase 17 pilot: four recipes and the selection (`gpu`; one cell; about 50 hours)

Trains `p17a`, `p17b`, `p17c` (30 steps each, `r17mix`'s mix, seed 1), picks `p17d`'s base (the better of `p17b` and `p17c` by `PILOT17_PLAN`'s rule) and trains `p17d` (40 turns), then prints the selection table (halted, invalid share over the last 10 steps, training welfare over steps 21-30, seconds per step) and the rule's choice. Each run in its own process; finished or halted runs are skipped after a disconnect. Writes `runs_p17pilot3/`, `selection.json` and `explRL_rl17pilot3.zip` (plan v3: the loss and KL on value tokens only, a per-token clipped ratio and a 3,072-token training window, after v2's KL halts; v1's and v2's runs stay in `runs_p17pilot/` and `runs_p17pilot2/`). If a run stops with CUDA out of memory (longer prompts now fit the window), report it before changing anything.

The pilot depends neither on Phase 16 nor on step 0. **Two accounts can share it:** run this cell on both (each with its own `HOST`); they claim `p17a`-`p17c` one each, whichever is free trains `p17d` once `p17b` and `p17c` are done, and the one that finishes last writes the selection and the zip (rerun the cell on any account when it says the pilot is not complete). A run folder with progress but no claim (started by an older cell) counts as held: to resume such a run, list it in `TAKEOVER` on the account that started it.

In [ ]:
#| gpu
from explRL.rl17_grid import PILOT17, PILOT17_PLAN, claim, pilotChoice
assert PILOT17_PLAN['frozen'], 'freeze, commit and push PILOT17_PLAN before the pilot'
WARM, P = OUT / 'warm_model', OUT / PILOT17_PLAN['dir']   # v3: runs_p17pilot3 (v1's and v2's runs stay)
done = lambda n: (P / n / 'final').exists()


def pilotRun(name, recipe=None):
    "Train one pilot run if it is not done and this account holds (or gets) its claim."
    if done(name):
        print(f'{name}: done')
        return
    if not claim(P / name, HOST, TAKEOVER):
        print(f'{name}: held by another account, skipped')
        return
    print(f'=== {name} on {HOST} ===', flush=True)
    rc = stream("from explRL.rl17_train import trainRL17; from explRL.rl17_grid import pilotConfig; "
                f"print(trainRL17(pilotConfig({name!r}, {str(WARM)!r}, {str(P / name)!r}, {recipe!r})))")
    if rc:
        raise RuntimeError(f'{name} failed (exit {rc}): ' + 'restart the session, run set-up and configuration, then this cell again (finished steps are skipped)')


def pilotMetrics(names):
    m = {n: [json.loads(x) for x in open(P / n / 'metrics.jsonl')] for n in names if (P / n / 'metrics.jsonl').exists()}
    return m, {n for n in names if (P / n / 'halted.json').exists()}


for n in ('p17a', 'p17b', 'p17c'):                           # two accounts split these by claims
    pilotRun(n)
bc = None
if done('p17b') and done('p17c'):
    bc, _ = pilotChoice(*pilotMetrics(['p17b', 'p17c']), names=['p17b', 'p17c'])
    if bc is None:
        print('neither p17b nor p17c passes the rule: p17d is not trained')
    else:
        pilotRun('p17d', PILOT17[bc])
names = ['p17a', 'p17b', 'p17c'] + (['p17d'] if bc else [])
if not all(done(n) for n in names) or (bc is None and not (done('p17b') and done('p17c'))):
    print('pilot not complete yet (runs on the other account, or p17d waits for p17b and p17c): rerun this cell later')
elif claim(P / '_select', HOST, TAKEOVER):
    choice, table = pilotChoice(*pilotMetrics(names), names=names)
    for n, t in table.items():
        print(f"{n}: halted {t['halted']}, invalid (last 10) {t['invalid']:.3f}, welfare (steps 21-30) {t['welfare']:.3f}, "
              f"{t['seconds']:.0f} s per step, cut prompts {t['cut']}, passes the rule: {t['ok']}")
    (P / 'selection.json').write_text(json.dumps(dict(choice=choice, p17dBase=bc, table=table), indent=1, default=str) + '\n')
    print('choice:', choice)
    packZip(PILOT17_PLAN['zip'], (PILOT17_PLAN['dir'],))

## Phase 17 main: claimed runs, then the evaluation (`gpu`; one cell per account)

Run this cell on every Colab account that shares the Drive folder (each with its own `HOST`). It trains every `RL17_RUNS` run that is not finished and not claimed by another account, each in its own process (an unfinished run resumes from its last checkpoint). When every run has its final adapter, one account wins the evaluation claim, evaluates all runs on `teamSizeSuite` and writes `explRL_rl17.zip`; rerun the cell on any account after the last run finishes. To move a run to another account: stop the first runtime, put the run's name in `TAKEOVER` on the other, rerun.

**Long runs:** with background execution the cell keeps going with the browser closed, but Colab ends a session after about 24 hours, and one run takes days. Runs checkpoint every step, so when a session ends: restart, set-up, configuration, this cell again; the run in progress resumes from its last step, finished runs are skipped, and the batch continues.

In [ ]:
#| gpu
from explRL.rl17_grid import RL17_PLAN, RL17_RUNS, claim
assert RL17_PLAN['frozen'] and RL17_PLAN['recipe'] and RL17_PLAN['userConfirmed'], \
    'freeze RL17_PLAN with the pilot-chosen recipe and the confirmed hours, commit and push first'
WARM, E17 = OUT / 'warm_model', OUT / 'eval_p17'
names = [f'{a}_s{s}' for a, s in RL17_RUNS]
for (arm, seed), name in zip(RL17_RUNS, names):
    d = RUNS17 / name
    if (d / 'final').exists():
        print(f'{name}: done')
        continue
    if not claim(d, HOST, TAKEOVER):
        print(f"{name}: held by {json.loads((d / 'claim.json').read_text())['host']}, skipped")
        continue
    print(f'=== {name} on {HOST} ===', flush=True)
    t0 = time.time()
    rc = stream("from explRL.rl17_train import trainRL17; from explRL.rl17_grid import RL17_PLAN, rl17Config; "
                f"print(trainRL17(rl17Config({arm!r}, {seed}, {str(WARM)!r}, {str(d)!r}, RL17_PLAN['recipe'])))")
    print(f'{name}: exit {rc}, {(time.time() - t0) / 3600:.2f} GPU hours', flush=True)
    if rc:
        raise RuntimeError(f'{name} failed (exit {rc}): ' + 'restart the session, run set-up and configuration, then this cell again (finished steps are skipped)')
pending = [n for n in names if not (RUNS17 / n / 'final').exists()]
if pending:
    print(f'waiting for {pending} (other accounts): rerun this cell after they finish')
elif not claim(E17, HOST, TAKEOVER):
    print(f"the evaluation is held by {json.loads((E17 / 'claim.json').read_text())['host']}")
else:
    models = {n: (str(WARM), str(RUNS17 / n / 'final')) for n in names}
    rc = stream("from explRL.bench_grid import writeCSV; from explRL.clue_roll import evalModels; from explRL.rl17_grid import teamSizeSuite; "
                f"rows = evalModels(teamSizeSuite({names!r}), {models!r}, {str(E17)!r}, cache='/content/evalcache', "
                f"logDir={str(E17 / 'turns')!r}); writeCSV(rows, {str(E17 / 'runs.csv')!r}); print(len(rows), 'boards')")
    if rc:
        raise RuntimeError(f'evaluation failed (exit {rc}): ' + 'restart the session, run set-up and configuration, then this cell again (finished steps are skipped)')
    packZip('explRL_rl17.zip', ('runs_p17', 'eval_p17'))
    print('Next, on the Mac: download explRL_rl17.zip, unzip into results/multiagent/rl/colab/, run demo_rl17 (flag train).')

## Phase 18 speed check (`gpu`; one account; about 1 hour)

`specs/multiagent/phase18/`, Part B. Three steps of `c18team` (team reward on the colony board, two holders per clue, Phase 17's frozen recipe and mix) with a 4,096-token window, into `speed_p18/`. Prints seconds per step, the longest prompt plus reply, cut prompts, the proposed training window (`maxLenRule`) and the projected GPU hours (`projectHours18`) for `RL18_PLAN`'s 30 steps (Part B lite, 5 runs): report them on the Mac, where the user confirms them and `RL18_PLAN` is frozen. Needs `RL17_PLAN` frozen with its recipe.

In [ ]:
#| gpu
from explRL.rl17_grid import RL17_PLAN
from explRL.rl18_grid import RL18_RUNS, RL18_STEPS, maxLenRule, projectHours18, speedConfig
assert RL17_PLAN['frozen'] and RL17_PLAN['recipe'], 'Part B needs Phase 17\'s frozen recipe'
WARM, SP = OUT / 'warm_model', OUT / 'speed_p18'
rc = stream("from explRL.rl17_train import trainRL17; from explRL.rl18_grid import speedConfig; from explRL.rl17_grid import RL17_PLAN; "
            f"print(trainRL17(speedConfig({str(WARM)!r}, {str(SP)!r}, RL17_PLAN['recipe'])))")
if rc:
    raise RuntimeError(f'speed check failed (exit {rc}): restart the session, run set-up and configuration, then this cell again')
m = [json.loads(x) for x in open(SP / 'metrics.jsonl')]
sec = sum(x['rolloutSeconds'] + x['updateSeconds'] for x in m[1:]) / max(1, len(m) - 1)   # step 1 includes engine start-up
longest, cut = max(x['longest'] for x in m), sum(x['cut'] for x in m)
h = projectHours18(sec, RL18_STEPS)
print(f"seconds per step {sec:.0f}; longest prompt plus reply {longest} tokens; cut prompts {cut}; invalid {[round(x['invalid'], 3) for x in m]}")
print(f"proposed maxLen {maxLenRule(longest)}; projected GPU hours at {RL18_STEPS} steps: {h['perRun']:.1f} per run, {h['total']:.0f} for all {len(RL18_RUNS)} runs")

## Phase 18 main: claimed runs, then the evaluation (`gpu`; one cell per account)

Run this cell on every Colab account that shares the Drive folder (each with its own `HOST`). It trains every `RL18_RUNS` run (Part B lite: `c18team`, 5 seeds, 30 steps) that is not finished and not claimed by another account, each in its own process (an unfinished run resumes from its last checkpoint). When every run has its final adapter, evaluation models (the warm start and the 5 adapters on `colonySuite`: the colony board, 8 and 32 agents with liars, 5 seeds) are claimed one at a time, so both accounts evaluate in parallel; the account that finds them all evaluated writes `runs.csv` and `explRL_rl18.zip`. Rerun the cell on any account when it says others still hold work. To move a run: stop the first runtime, add the run's name to `TAKEOVER` on the other, rerun. The same 24-hour session rule as Phase 17.

In [ ]:
#| gpu
from explRL.rl17_grid import RL17_PLAN, claim
from explRL.rl18_grid import RL18_PLAN, RL18_RUNS, adapterReady, rl18Config, runDone
assert RL18_PLAN['frozen'] and RL18_PLAN['userConfirmed'] and RL18_PLAN['steps'] and RL18_PLAN['maxLen'], \
    'freeze RL18_PLAN with the steps, window and confirmed hours (after the speed check), commit and push first'
WARM, E18 = OUT / 'warm_model', OUT / 'eval_p18'
names = [f'{a}_s{s}' for a, s in RL18_RUNS]
for (arm, seed), name in zip(RL18_RUNS, names):
    d = RUNS18 / name
    if runDone(d):                                          # halted, or the final adapter's weights saved (not just the folder)
        print(f'{name}: done')
        continue
    if not claim(d, HOST, TAKEOVER):
        print(f"{name}: held by {json.loads((d / 'claim.json').read_text())['host']}, skipped")
        continue
    print(f'=== {name} on {HOST} ===', flush=True)
    t0 = time.time()
    rc = stream("from explRL.rl17_train import trainRL17; from explRL.rl18_grid import RL18_PLAN, rl18Config; from explRL.rl17_grid import RL17_PLAN; "
                f"print(trainRL17(rl18Config({arm!r}, {seed}, {str(WARM)!r}, {str(d)!r}, RL17_PLAN['recipe'], RL18_PLAN['steps'], RL18_PLAN['maxLen'])))")
    print(f'{name}: exit {rc}, {(time.time() - t0) / 3600:.2f} GPU hours', flush=True)
    if rc:
        raise RuntimeError(f'{name} failed (exit {rc}): restart the session, run set-up and configuration, then this cell again (finished steps are skipped)')
pending = [n for n in names if not runDone(RUNS18 / n)]
if pending:
    print(f'waiting for {pending} (other accounts): rerun this cell after they finish')
else:
    from explRL.rl18_grid import evalAdapter               # a halted run: the checkpoint before its halting streak (RL18_PLAN)
    models = {'warm': (str(WARM), None), **{n: (str(WARM), evalAdapter(RUNS18 / n)) for n in names}}
    print('evaluated adapters:', {n: v[1] for n, v in models.items()})
    for mname, v in models.items():                         # one model at a time, claimed: two accounts share the evaluation
        if v[1] and not adapterReady(v[1]):                # Drive has not delivered the weights here yet
            print(f'{mname}: adapter weights not in {v[1]} yet (still syncing from the other account?): rerun this cell later')
            continue
        if (E18 / f'{mname}.csv').exists() or not claim(E18 / 'claims' / mname, HOST, TAKEOVER):
            continue
        print(f'=== evaluating {mname} on {HOST} ===', flush=True)
        rc = stream("from explRL.clue_roll import evalModels; from explRL.rl18_grid import colonySuite; "
                    f"evalModels(colonySuite([{mname!r}]), {{{mname!r}: {v!r}}}, {str(E18)!r}, cache='/content/evalcache', logDir={str(E18 / 'turns')!r})")
        if rc:
            raise RuntimeError(f'evaluation of {mname} failed (exit {rc}): restart the session, run set-up and configuration, then this cell again')
    left = [m for m in models if not (E18 / f'{m}.csv').exists()]
    if left:
        print(f'evaluation not complete yet: {left} (other account): rerun this cell later')
    elif claim(E18 / 'claims' / '_pack', HOST, TAKEOVER):
        assert stream("from explRL.bench_grid import writeCSV; from explRL.clue_roll import evalModels; from explRL.rl18_grid import colonySuite; "
                      f"rows = evalModels(colonySuite({list(models)!r}), {models!r}, {str(E18)!r}); "
                      f"writeCSV(rows, {str(E18 / 'runs.csv')!r}); print(len(rows), 'boards')") == 0       # all read back
        packZip('explRL_rl18.zip', ('runs_p18', 'eval_p18'))
        print('Next, on the Mac: download explRL_rl18.zip, unzip into results/multiagent/rl/colab/, run demo_rl18 (flag train).')

## Pack (`gpu`): adapters, metrics and configurations for the Mac

In [ ]:
#| gpu
import zipfile
with zipfile.ZipFile(OUT / 'explRL_rl.zip', 'w') as z:
    for p in sorted(OUT.rglob('*')):
        rel = p.relative_to(OUT)
        if p.is_file() and (rel.parts[0] in ('warm', 'pilot', 'speedcheck', 'lrpilot', 'eval', 'eval_p13', 'eval_p16', 'eval_p17s0', 'eval_p17') or rel.parts[0].startswith('runs')) and 'ckpt-' not in str(rel) and rel.parts[:2] != ('pilot', 'warm'):
            z.write(p, rel)
print(OUT / 'explRL_rl.zip', round((OUT / 'explRL_rl.zip').stat().st_size / 1e6, 1), 'MB')